In [1]:
import numpy as np

from tensorflow.python.keras.models import Sequential
from tensorflow.python.keras.layers import Dense, Activation, Dropout
from tensorflow.python.keras.losses import CategoricalCrossentropy
from tensorflow.python.keras.optimizers import adam_v2

2023-07-02 20:31:01.229467: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-07-02 20:31:01.343619: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-07-02 20:31:01.344902: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-02 20:31:02.393719: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
config = {
      "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 20,
    "batch_size": 128,
    "loss": CategoricalCrossentropy(),
    "optimizer": adam_v2.Adam(learning_rate=0.001),
}

In [6]:
from keras.datasets import mnist
from keras.utils import to_categorical

(X_train, y_train),(X_test, y_test) = mnist.load_data()
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

image_size = X_train.shape[1]
input_size = image_size * image_size

X_train = np.reshape(X_train, [-1, input_size])
X_train = X_train.astype('float32') / 255
X_test = np.reshape(X_test, [-1, input_size])
X_test = X_test.astype('float32') / 255

np.save('../../../data/MNIST/train_data.npy', X_train)
np.save('../../../data/MNIST/train_labels.npy', y_train)
np.save('../../../data/MNIST/test_data.npy', X_test)
np.save('../../../data/MNIST/test_labels.npy', y_test)

In [10]:
def get_train_data():
    return np.load('../../../data/MNIST/train_data.npy'), np.load('../../../data/MNIST/train_labels.npy')

def get_test_data():
    return np.load('../../../data/MNIST/test_data.npy'), np.load('../../../data/MNIST/test_labels.npy')

In [11]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation('relu'))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation('softmax'))
    return model

def train_model(model, X_train, y_train, loss, optimizer, epochs, batch_size):
    model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size)
    return model

In [12]:
X_train, y_train = get_train_data()

In [7]:
model = create_model()
model = train_model(model, X_train, y_train, config["loss"], config["optimizer"], config["epochs"], config["batch_size"])

2023-06-30 21:36:46.683808: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 188160000 exceeds 10% of free system memory.


Epoch 1/20
469/469 [==============================] - 4s 7ms/step - loss: 0.4259 - accuracy: 0.8703
Epoch 2/20
469/469 [==============================] - 3s 6ms/step - loss: 0.1921 - accuracy: 0.9428
Epoch 3/20
469/469 [==============================] - 3s 6ms/step - loss: 0.1523 - accuracy: 0.9546
Epoch 4/20
469/469 [==============================] - 3s 6ms/step - loss: 0.1305 - accuracy: 0.9613
Epoch 5/20
469/469 [==============================] - 3s 6ms/step - loss: 0.1138 - accuracy: 0.9648
Epoch 6/20
469/469 [==============================] - 3s 6ms/step - loss: 0.1047 - accuracy: 0.9682
Epoch 7/20
469/469 [==============================] - 3s 6ms/step - loss: 0.0959 - accuracy: 0.9707
Epoch 8/20
469/469 [==============================] - 3s 6ms/step - loss: 0.0910 - accuracy: 0.9714
Epoch 9/20
469/469 [==============================] - 3s 7ms/step - loss: 0.0806 - accuracy: 0.9746
Epoch 10/20
469/469 [==============================] - 3s 6ms/step - loss: 0.0784 - accuracy: 0.9750

In [8]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 3ms/step - loss: 0.0651 - accuracy: 0.9820

Test accuracy: 98.2%
